# Phase 1 Chunk 4:

## First Neural Network - Binary Classifier

### 1. Dataset-Centric Introduction: Breast Cancer Wisconsin Dataset
**Problem Context**: You are a medical researcher with a dataset of 569 tumor biopsies. For each tumor, you have 30 different measurements (e.g., mean radius, mean texture, mean smoothness). Your critical task is to build a model that can predict whether a tumor is malignant (cancerous) or benign (non-cancerous).

**Why it Matters**: This is a classic and impactful binary classification problem. The stakes are high, and it's a perfect real-world example of how a simple neural network can learn complex patterns from tabular data. It forces us to handle data splitting (training vs. testing) and understand the output of a classification model.

### 2. Theory-to-Practice Bridge: From a Single Neuron to a nn.Module
In the last chunk, you manually defined w and b. This is not scalable. Imagine a network with millions of parameters!
PyTorch's torch.nn.Module is the solution. It's a base class that all our models will inherit from. It does two key things:

* Organizes Layers: We define our layers (like nn.Linear) in the __init__ method.

* Defines the Forward Pass: We specify how data flows through these layers in the forward method.

### Theory Connection:
**nn.Linear(in_features, out_features)**: This is your dense or fully-connected layer. It encapsulates the y = xW^T + b operation. It automatically creates and initializes the weight W and bias b tensors for you.

**torch.sigmoid**: This is the sigmoid activation function you learned about. It squashes the output of our linear layer to be between 0 and 1, which we can interpret as a probability. probability = 1 / (1 + e^(-z)).

`Crucially, nn.Module automatically tracks all the parameters within its layers. When you move the model to a GPU (model.to(device)), all its parameters move. When you call model.parameters(), it gives you a list of all learnable w's and b's for the optimizer.`

In [2]:
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Reproducibility
torch.manual_seed(42)

data = load_breast_cancer()
X,y = data.data, data.target

# Splitting the dataset

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale your dat. It is a Professional best practice.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to Tensors

X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).view(-1,1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32, device=device).view(-1,1)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}") 


X_train_tensor shape: torch.Size([455, 30])
y_train_tensor shape: torch.Size([455, 1])


## ___Define the Neural Network Class___

### __`We are building a simple Logistic regression model, which is a `1` Layer NN`__

In [23]:
class BinaryClassifier(nn.Module):
    def __init__(self, num_features):
        super(BinaryClassifier, self).__init__()
        # Theory-to-practice: This is our single layer.
        # It takes `num_features` inputs ( 30 in our case ) and produces 1 output
        self.linear = nn.Linear(num_features, 1, device=device)
    
    def forward(self, x):
        # Theory-to-practice: This defines the data flow.
        # 1. pass through the linear layer
        # 2. Apply sigmoid activation to get a probability
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred
    

####
# Instantiate the model
num_features = X_train.shape[1] # 30 features
torch.tensor(num_features).to(device)
model = BinaryClassifier(num_features)
print("-----_-_-_-_-_     Model Architecture     _-_-_-_-_-----")
print(" ")
print(model)




# Let's inspect the randomly initialized weights
print("\nInitial Weights:", model.linear.weight)
print("Initial Bias:", model.linear.bias)


#      Test a forward pass

# Let's pass the first 5 samples through the untrained model
with torch.no_grad():  # Because we don't need gradients when doing inference
    y_pred_test = model(X_test_tensor[:5])


print("\nUNtrained Model Predictions")
print(f"Inpute shape : {X_train[:5].shape}")
print(f"Prediction Shape : {y_pred_test.shape}")
print(f"\nPredictions ( Probabilities )")
print(y_pred_test.squeeze())
print("\nActual Labels:")
print(y_test_tensor[:5].squeeze())

# The predictions are close to 0.5 because the model is initialized randomly.
# It hasn't learned anything yet!

-----_-_-_-_-_     Model Architecture     _-_-_-_-_-----
 
BinaryClassifier(
  (linear): Linear(in_features=30, out_features=1, bias=True)
)

Initial Weights: Parameter containing:
tensor([[-0.0880, -0.1792, -0.1082, -0.1067, -0.1076,  0.0782, -0.1774,  0.1496,
         -0.1395, -0.0075,  0.0422,  0.0546,  0.0245, -0.1561, -0.1308,  0.1566,
          0.1616, -0.0122, -0.0776, -0.1671, -0.0655, -0.0413,  0.0393,  0.0339,
         -0.0387,  0.0976,  0.0810,  0.0672,  0.1800,  0.0113]],
       device='cuda:0', requires_grad=True)
Initial Bias: Parameter containing:
tensor([0.0887], device='cuda:0', requires_grad=True)

UNtrained Model Predictions
Inpute shape : (5, 30)
Prediction Shape : torch.Size([5, 1])

Predictions ( Probabilities )
tensor([0.5270, 0.4597, 0.4623, 0.4984, 0.5541], device='cuda:0')

Actual Labels:
tensor([1., 0., 0., 1., 1.], device='cuda:0')


# Explorations


## Multi-Level Explorations
**Beginner**: 

Print the `model.parameters()` right after you create the model. What are the shapes of the weight and bias tensors inside the linear layer? Do they match what you expect for a model with 30 inputs and 1 output?

**Intermediate**: 

Add a hidden layer to the `BinaryClassifier`. Modify the __init__ to have two `nn.Linear` layers: self.layer1 = nn.Linear(num_features, 10) and self.layer2 = nn.Linear(10, 1). Then, modify the forward method to pass the data through layer1, apply a `torch.relu` activation, and then pass that result through layer2 before the final sigmoid. This creates a deeper network!

**Advanced**: 

`nn.Module` has a powerful feature called **`hooks`**. A forward hook is a function that gets executed after a forward pass on a specific layer. Register a forward hook on `model.linear` that prints the shape of the layer's input and output tensors. This is an incredibly useful technique for debugging complex architectures. `(Hint: handle = model.linear.register_forward_hook(my_hook_function)).`

## 5. Dataset-Specific Exercises

**Replication**: 
* Replicate this exact example using the sklearn.datasets.make_classification dataset we used in Chunk 2. Ensure you set it up for binary classification (n_classes=2). Remember to scale your data and split it into training and test sets.

**Modification**: 
* The output of our model is a probability. To make a final decision (Malignant or Benign), we need a threshold (typically 0.5). Using the y_pred_test from the example code, convert the probabilities into binary predictions (0 or 1). Then, calculate the accuracy of the (untrained) model on those first 5 samples.

**Creation**: 
* Build a new nn.Module for a regression task. Use the California Housing dataset. Your model should take in the 8 features of the housing data and output a single continuous value (the predicted house price). What activation function should you use on the final layer for a regression problem like this? (Hint: it's not sigmoid).

## Professional Best Practices
**train_test_split**: 
* NEVER train your model on your test data. Splitting your data is the first and most crucial step in any legitimate machine learning project to get an unbiased estimate of your model's performance.

**Data Scaling (StandardScaler)**: 
* Features in real-world datasets often have vastly different scales (e.g., age in years, income in thousands of dollars). Scaling them to have a mean of 0 and a standard deviation of 1 (StandardScaler) helps the model's optimization process converge much faster and more reliably.

**with torch.no_grad()**: 
* When you are only doing inference (making predictions) and not training, you should wrap your code in this context manager. It tells PyTorch not to build the computation graph, which saves a significant amount of memory and computation time.